# Step 2 - Does the training loop run?

Before we put the real reward in, we check the training machinery runs at all.
This trains for just a few steps with a FAKE reward (a random number). We are
not looking for the model to get better. We only want it to run start to finish
without crashing.

If this works, the risky part of the project is behind us.

## Setup: get the repo

In [ ]:
import os, sys, subprocess

REPO = "https://github.com/ookino/rlvr-argument-mining.git"
NAME = "rlvr-argument-mining"

if not os.path.exists("reward"):
    if not os.path.exists(NAME):
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir(NAME)
else:
    subprocess.run(["git", "pull", "--quiet"], check=False)

sys.path.insert(0, os.getcwd())
print("repo ready")

## Install the training libraries

`unsloth` loads the model cheaply in 4-bit and gives us fast GRPO. It pulls in
`trl` (which has the GRPO trainer), `transformers`, `peft` and the rest. This
cell takes a few minutes the first time.

In [ ]:
!pip install -q unsloth

## Print the library versions

If the training cell later fails, these versions are the first thing to check,
because the GRPO library changes its function names between versions.

In [ ]:
import unsloth          # import first so it can patch the others
import torch, transformers, trl
print("unsloth     ", unsloth.__version__)
print("trl         ", trl.__version__)
print("transformers", transformers.__version__)
print("torch       ", torch.__version__)
print("gpu         ", torch.cuda.is_available())

## Run the smoke test

This loads Qwen 2.5 3B in 4-bit, adds the small trainable adapters, and runs 5
GRPO steps on a handful of toy questions with a random reward. Loading the model
takes a minute. The 5 steps are slow because the model writes several answers
per question.

Success looks like a short training table and the line:

    training loop finished without crashing

In [ ]:
from train.grpo_train import run

trainer = run("configs/baseline.yaml", max_steps=5)

## What to do next

- If it printed **training loop finished without crashing**: step 2 is done.
  Tell Claude and we wire in the real reward.
- If it **errored**: copy the whole error and paste it back. Version mismatches
  in the GRPO library are common and quick to fix once we see the message.